In [1]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# %                   SCF-TB - PROXY APPLICATION                      %
# %        A.M.N. Niklasson, A.P. Baldo, M. Kulichenko. T1, LANL      %
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# % Total Energy Function:                                            %
# % E = 2Tr[H0(D-D0)] + (1/2)*sum_i U_i q_i^2 +                       %
# %      + (1/2)sum_{i,j (i!=j)} q_i C_{ij} q_j - Efield*dipole       %
# % dipole = sum_i R_{i} q_i                                          %
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

import torch
import warnings
import logging
import os

# to disable torchdynamo completely. Faster for smaller systems and single-point calculations.
os.environ["TORCHDYNAMO_DISABLE"] = "1"  # hard-disable capture


from dftorch.Constants import Constants
from dftorch.Structure import Structure
from dftorch.ESDriver import ESDriver

### Configure torch and torch.compile ###
# Silence warnings and module logs
warnings.filterwarnings("ignore")
os.environ["TORCH_LOGS"] = ""  # disable PT2 logging
os.environ["TORCHINDUCTOR_VERBOSE"] = "0"
os.environ["TORCHDYNAMO_VERBOSE"] = "0"
logging.getLogger("torch.fx").setLevel(logging.CRITICAL)
logging.getLogger("torch.fx.experimental.symbolic_shapes").setLevel(logging.CRITICAL)
logging.getLogger("torch.fx.experimental.recording").setLevel(logging.CRITICAL)
# Enable dynamic shape capture for dynamo
torch._dynamo.config.capture_dynamic_output_shape_ops = True
# default data type
torch.set_default_dtype(torch.float64)

torch.cuda.empty_cache()

# unrestricted GS Singlet for acetone in vacuum

In [ ]:
dftorch_params = {
    "FILENAME": "COORD_ACETONE.xyz",
    "CELL": [12.0, 12.0, 12.0],  # Simulation box vectors in Angstroms
    "SKFPATH": "sk_orig/mio-1-1/mio-1-1/",  # Path to SKF files
    "T_ELECTRONIC": 300.0,  # Electronic temperature in Kelvin for Fermi smearing
    "RCUT_ELECTRONIC": 8.0,  # Cutoff for electronic interactions in Angstroms. Should be >= largest cutoff in SKF files for the element pairs present in the system.
    "RCUT_REPULSIVE": 4.0,  # Cutoff for repulsive interactions in Angstroms. Should be >= largest cutoff in SKF files for the element pairs present in the system.
    "COUL_METHOD": "!PME",  # 'FULL' for full coulomb matrix, 'PME' for Particle Mesh Ewald method
    "COULOMB_ACC": 1e-6,  # Coulomb accuracy for full coulomb calcs or t_err for PME
    "COULOMB_CUTOFF": 12.0,  # Coulomb cutoff

    "UNRESTRICTED": True,

    ########   Delta SCF parameters   ##################
    "DELTA_SCF": False,  # if True, perform delta SCF for targeted, non-aufbau excited state. Performs GS SCF, then ES SCF.
    "DELTA_SCF_TARGET": "SINGLET",  # options: '"SINGLET" or "TRIPLET"'. desired lowest excited state
    "DELTA_SCF_SMEARING": False,  # if True, occupations for GS orbital and target ES orbital will be set to 0.5
    ####################################################
    "KRYLOV_MAXRANK": 3,  # Maximum Krylov subspace rank
    "KRYLOV_TOL_MD": 1e-6,  # Krylov subspace convergence tolerance in MD SCF
    "KRYLOV_START": 3,  # Number of initial SCF iterations before starting Krylov acceleration
}

# Initial data, load atoms and coordinates, etc in COORD.dat
device = "cuda" if torch.cuda.is_available() else "cpu"


In [4]:
# Create constants container. Set path to SKF files.
const = Constants(dftorch_params).to(device)

structure1 = Structure(dftorch_params, const, device=device)

# Create ESDriver object and run SCF calculation
# electronic_rcut and repulsive_rcut are in Angstroms.
# They should be >= cutoffs defined in SKF files for the element pair with largest cutoff present in the system.
es_driver = ESDriver(dftorch_params, device=device)
es_driver(structure1, const, do_scf=True)
es_driver.calc_forces(structure1, const)  # Calculate forces after SCF

DFTB3: False
coulomb_matrix_vectorized
  Coulomb_Real t 0.0 s
   LMAX: 5
  Coulomb_k t 0.2 s

### Do _scf ###
  Initial dm_fermi

Starting cycle
Iter 1
Res = 0.530460970, dEc = 0.920783816, t = 0.0 s

Iter 2
Res = 0.301705345, dEc = 0.472487559, t = 0.0 s

Iter 3
Res = 0.022658882, dEc = 0.054970562, t = 0.0 s

Iter 4
  rank: 0, Fel = 0.000035
  rank: 1, Fel = 0.000005
  rank: 2, Fel = 0.000001
Res = 0.026109326, dEc = 0.033152338, t = 0.0 s

Iter 5
  rank: 0, Fel = 0.000001
Res = 0.000078977, dEc = 0.000088214, t = 0.0 s

Iter 6
  rank: 0, Fel = 0.000000
Res = 0.000007284, dEc = 0.000007959, t = 0.0 s

Iter 7
  rank: 0, Fel = 0.000000
Res = 0.000003438, dEc = 0.000002738, t = 0.0 s

Iter 8
  rank: 0, Fel = 0.000000
Res = 0.000000490, dEc = 0.000000573, t = 0.0 s



In [5]:
print("occupancies: alpha     beta")
print(structure1.f.t())
print("electronic free energy (eV):")
print(structure1.e_elec_tot)

occupancies: alpha     beta
tensor([[-6.8902e-16, -6.8902e-16],
        [-3.3870e-16, -3.3870e-16],
        [-4.6296e-17, -4.6296e-17],
        [-2.6376e-17, -2.6376e-17],
        [ 5.6251e-17,  5.6251e-17],
        [ 7.8997e-17,  7.8997e-17],
        [ 1.2784e-16,  1.2784e-16],
        [ 2.3457e-16,  2.3457e-16],
        [ 4.9265e-16,  4.9265e-16],
        [ 5.8524e-16,  5.8524e-16],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00]], device='cuda:0')
electronic free energy (eV):
tensor(-61.4301, device='cuda:0')


# Lowest ES Singlet for acetone in vacuum (Half Excitation)

In [6]:
dftorch_params = {
    "FILENAME": "COORD_ACETONE.xyz",
    "CELL": [12.0, 12.0, 12.0],  # Simulation box vectors in Angstroms
    "SKFPATH": "sk_orig/mio-1-1/mio-1-1/",  # Path to SKF files
    "T_ELECTRONIC": 300.0,  # Electronic temperature in Kelvin for Fermi smearing
    "RCUT_ELECTRONIC": 8.0,  # Cutoff for electronic interactions in Angstroms. Should be >= largest cutoff in SKF files for the element pairs present in the system.
    "RCUT_REPULSIVE": 4.0,  # Cutoff for repulsive interactions in Angstroms. Should be >= largest cutoff in SKF files for the element pairs present in the system.
    "COUL_METHOD": "!PME",  # 'FULL' for full coulomb matrix, 'PME' for Particle Mesh Ewald method
    "COULOMB_ACC": 1e-6,  # Coulomb accuracy for full coulomb calcs or t_err for PME
    "COULOMB_CUTOFF": 12.0,  # Coulomb cutoff

    "UNRESTRICTED": True,

    ########   Delta SCF parameters   ##################
    "DELTA_SCF": True,  # if True, perform delta SCF for targeted, non-aufbau excited state. Performs GS SCF, then ES SCF.
    "DELTA_SCF_TARGET": "SINGLET",  # options: '"SINGLET" or "TRIPLET"'. desired lowest excited state
    "DELTA_SCF_SMEARING": True,  # if True, occupations for GS orbital and target ES orbital will be set to 0.5
    ####################################################
    "KRYLOV_MAXRANK": 3,  # Maximum Krylov subspace rank
    "KRYLOV_TOL_MD": 1e-6,  # Krylov subspace convergence tolerance in MD SCF
    "KRYLOV_START": 3,  # Number of initial SCF iterations before starting Krylov acceleration
}

device = "cuda" if torch.cuda.is_available() else "cpu"
const = Constants(dftorch_params).to(device)
structure1 = Structure(dftorch_params, const, device=device)

es_driver = ESDriver(dftorch_params, device=device)
es_driver(structure1, const, do_scf=True)
es_driver.calc_forces(structure1, const)  # Calculate forces after SCF

DFTB3: False
coulomb_matrix_vectorized
  Coulomb_Real t 0.0 s
   LMAX: 5
  Coulomb_k t 0.2 s

### Do _scf ###
  Initial dm_fermi

Starting cycle
Iter 1
Res = 0.530460970, dEc = 0.920783816, t = 0.0 s

Iter 2
Res = 0.301705345, dEc = 0.472487559, t = 0.0 s

Iter 3
Res = 0.022658882, dEc = 0.054970562, t = 0.0 s

Iter 4
  rank: 0, Fel = 0.000035
  rank: 1, Fel = 0.000005
  rank: 2, Fel = 0.000001
Res = 0.026109326, dEc = 0.033152338, t = 0.0 s

Iter 5
  rank: 0, Fel = 0.000001
Res = 0.000078977, dEc = 0.000088214, t = 0.0 s

Iter 6
  rank: 0, Fel = 0.000000
Res = 0.000007284, dEc = 0.000007959, t = 0.0 s

Iter 7
  rank: 0, Fel = 0.000000
Res = 0.000003438, dEc = 0.000002738, t = 0.0 s

Iter 8
  rank: 0, Fel = 0.000000
Res = 0.000000490, dEc = 0.000000573, t = 0.0 s

### Do Delta_scf ###
  Initial dm_fermi

Starting cycle
Iter 1
Res = 0.158029844, dEc = 0.183072351, t = 0.0 s

Iter 2
Res = 0.129647805, dEc = 0.015901300, t = 0.0 s

Iter 3
Res = 0.107124211, dEc = 0.013267360, t = 0.0 s

I

In [7]:
print("occupancies: alpha     beta")
print(structure1.f.t())
print("electronic free energy (eV):")
print(structure1.e_elec_tot)

occupancies: alpha     beta
tensor([[ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  5.0000e-01],
        [ 8.8177e-44,  5.0000e-01],
        [7.8526e-197, 3.4019e-196],
        [6.0504e-208, 1.4716e-207],
        [1.2646e-209, 3.6925e-209],
        [3.0163e-219, 1.2675e-218],
        [1.1418e-239, 4.4620e-238],
        [7.1287e-259, 6.8187e-257],
        [ 0.0000e+00, 6.1598e-308],
        [ 0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00]], device='cuda:0')
electronic free energy (eV):
tensor(-59.0498, device='cuda:0')


# Lowest ES Singlet for acetone in vacuum (Full Excitation)

In [8]:
dftorch_params = {
    "FILENAME": "COORD_ACETONE.xyz",
    "CELL": [12.0, 12.0, 12.0],  # Simulation box vectors in Angstroms
    "SKFPATH": "sk_orig/mio-1-1/mio-1-1/",  # Path to SKF files
    "T_ELECTRONIC": 300.0,  # Electronic temperature in Kelvin for Fermi smearing
    "RCUT_ELECTRONIC": 8.0,  # Cutoff for electronic interactions in Angstroms. Should be >= largest cutoff in SKF files for the element pairs present in the system.
    "RCUT_REPULSIVE": 4.0,  # Cutoff for repulsive interactions in Angstroms. Should be >= largest cutoff in SKF files for the element pairs present in the system.
    "COUL_METHOD": "!PME",  # 'FULL' for full coulomb matrix, 'PME' for Particle Mesh Ewald method
    "COULOMB_ACC": 1e-6,  # Coulomb accuracy for full coulomb calcs or t_err for PME
    "COULOMB_CUTOFF": 12.0,  # Coulomb cutoff

    "UNRESTRICTED": True,

    ########   Delta SCF parameters   ##################
    "DELTA_SCF": True,  # if True, perform delta SCF for targeted, non-aufbau excited state. Performs GS SCF, then ES SCF.
    "DELTA_SCF_TARGET": "SINGLET",  # options: '"SINGLET" or "TRIPLET"'. desired lowest excited state
    "DELTA_SCF_SMEARING": False,  # if True, occupations for GS orbital and target ES orbital will be set to 0.5
    ####################################################
    "KRYLOV_MAXRANK": 15,  # Maximum Krylov subspace rank
    "KRYLOV_TOL_MD": 1e-6,  # Krylov subspace convergence tolerance in MD SCF
    "KRYLOV_START": 3,  # Number of initial SCF iterations before starting Krylov acceleration
}

device = "cuda" if torch.cuda.is_available() else "cpu"
const = Constants(dftorch_params).to(device)
structure1 = Structure(dftorch_params, const, device=device)

es_driver = ESDriver(dftorch_params, device=device)
es_driver(structure1, const, do_scf=True)
es_driver.calc_forces(structure1, const)  # Calculate forces after SCF

DFTB3: False
coulomb_matrix_vectorized
  Coulomb_Real t 0.0 s
   LMAX: 5
  Coulomb_k t 0.2 s

### Do _scf ###
  Initial dm_fermi

Starting cycle
Iter 1
Res = 0.530460970, dEc = 0.920783816, t = 0.0 s

Iter 2
Res = 0.301705345, dEc = 0.472487559, t = 0.0 s

Iter 3
Res = 0.022658882, dEc = 0.054970562, t = 0.0 s

Iter 4
  rank: 0, Fel = 0.000035
  rank: 1, Fel = 0.000005
  rank: 2, Fel = 0.000001
Res = 0.026109326, dEc = 0.033152338, t = 0.0 s

Iter 5
  rank: 0, Fel = 0.000001
Res = 0.000078977, dEc = 0.000088214, t = 0.0 s

Iter 6
  rank: 0, Fel = 0.000000
Res = 0.000007284, dEc = 0.000007959, t = 0.0 s

Iter 7
  rank: 0, Fel = 0.000000
Res = 0.000003438, dEc = 0.000002738, t = 0.0 s

Iter 8
  rank: 0, Fel = 0.000000
Res = 0.000000490, dEc = 0.000000573, t = 0.0 s

### Do Delta_scf ###
  Initial dm_fermi

Starting cycle
Iter 1
Res = 0.325723380, dEc = 0.110393947, t = 0.0 s

Iter 2
Res = 0.260545031, dEc = 0.010222366, t = 0.0 s

Iter 3
Res = 0.211274719, dEc = 0.010608486, t = 0.0 s

I

In [9]:
print("occupancies: alpha     beta")
print(structure1.f.t())
print("electronic free energy (eV):")
print(structure1.e_elec_tot)

occupancies: alpha     beta
tensor([[ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  1.0000e+00],
        [ 1.0000e+00,  0.0000e+00],
        [ 1.7798e-47,  1.0000e+00],
        [2.7930e-198, 6.6473e-198],
        [2.0587e-209, 1.7675e-209],
        [8.3130e-211, 9.4236e-211],
        [5.5517e-221, 1.4763e-220],
        [3.2948e-242, 7.8587e-240],
        [9.8336e-262, 1.2634e-258],
        [ 0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00]], device='cuda:0')
electronic free energy (eV):
tensor(-56.4583, device='cuda:0')
